# Polarization example - maximum likelihood method

This notebook fits the polarization fraction and angle of a Data Challenge 3 GRB (GRB 080802386) simulated using MEGAlib and combined with albedo photon background. It's assumed that the start time, duration, localization, and spectrum of the GRB are already known. The GRB was simulated with 80% polarization at an angle of 90 degrees in the IAU convention, and was 20 degrees off-axis. 

In [1]:
%%capture
from cosipy import BinnedData
from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.statistics import PoissonLikelihood
from cosipy.background_estimation import FreeNormBinnedBackground
from cosipy.interfaces import ThreeMLPluginInterface
from cosipy.response.FullDetectorResponse import FullDetectorResponse
from cosipy.response import BinnedThreeMLModelFolding, BinnedInstrumentResponse, BinnedThreeMLPointSourceResponse
from cosipy.data_io import EmCDSBinnedData
from cosipy.threeml.custom_functions import Band_Eflux
from cosipy.polarization import PolarizationAxis
from cosipy.sensitivity.mdp import compute_mdp
from astropy.time import Time
from astropy.coordinates import SkyCoord
from astropy import units as u
from cosipy.util import fetch_wasabi_file
from pathlib import Path
import sys
from threeML import LinearPolarization, StokesPolarization, SpectralComponent, PointSource, Model, JointLikelihood, DataList
from astromodels import Parameter, Constant
import numpy as np
from cosipy.sensitivity.mdp import compute_mdp
from cosipy.threeml.util import to_linear_polarization

### Download and read in data

This will download the files needed to run this notebook. If you have already downloaded these files, you can skip this.

Download the unbinned data (660.58 KB), polarization response (217.47 MB), and orientation file (1.10 GB)

In [4]:
fetch_wasabi_file('COSI-SMEX/cosipy_tutorials/polarization_fit/grb_background.fits.gz', checksum = '21b1d75891edc6aaf1ff3fe46e91cb49')
fetch_wasabi_file('COSI-SMEX/develop/Data/Responses/ResponseContinuum.o3.pol.e200_10000.b4.p12.relx.s10396905069491.m420.filtered.binnedpolarization.11D.h5', checksum = 'd417d241be0ba7fbfeaef5e2c3bd6ebc')
fetch_wasabi_file('COSI-SMEX/DC4/Data/Orientation/DC4_final_530km_3_month_with_slew_1sbins_GalacticEarth_SAA.fits', checksum = '1b851c042acf4c909798e2401e9d2e38')

A file named grb_background.fits.gz already exists with the specified checksum (21b1d75891edc6aaf1ff3fe46e91cb49). Skipping.
A file named ResponseContinuum.o3.pol.e200_10000.b4.p12.relx.s10396905069491.m420.filtered.binnedpolarization.11D.h5 already exists with the specified checksum (d417d241be0ba7fbfeaef5e2c3bd6ebc). Skipping.
A file named DC4_final_530km_3_month_with_slew_1sbins_GalacticEarth_SAA.fits already exists with the specified checksum (1b851c042acf4c909798e2401e9d2e38). Skipping.


Read in and bin the data, which is a GRB placed within albedo photon background. A time cut is done for the duration of the GRB to produce the GRB+background data to fit. The time intervals before and after the GRB are used to produce a background model.

In [ ]:
data_path = Path('') # Update to your path

grb_background = BinnedData(data_path/'grb.yaml')
grb_background.select_data_time(unbinned_data=data_path/'grb_background.fits.gz', output_name=data_path/'grb_background_source_interval') 
grb_background.get_binned_data(unbinned_data=data_path/'grb_background_source_interval.fits.gz', output_name=data_path/'grb_background_binned_galactic', psichi_binning='galactic')
grb_background.load_binned_data_from_hdf5(data_path/'grb_background_binned_galactic.hdf5')

background_before = BinnedData(data_path/'background_before.yaml')
background_before.select_data_time(unbinned_data=data_path/'grb_background.fits.gz', output_name=data_path/'background_before')
background_before.get_binned_data(unbinned_data='background_before.fits.gz', output_name='background_before_binned_galactic', psichi_binning='galactic')
background_before.load_binned_data_from_hdf5(data_path/'background_before_binned_galactic.hdf5')

background_after = BinnedData(data_path/'background_after.yaml') # e.g. background_after.yaml
background_after.select_data_time(unbinned_data=data_path/'grb_background.fits.gz', output_name=data_path/'background_after')
background_after.get_binned_data(unbinned_data=data_path/'background_after.fits.gz', output_name=data_path/'background_after_binned_galactic', psichi_binning='galactic')
background_after.load_binned_data_from_hdf5(data_path/'background_after_binned_galactic.hdf5')

fill() discarded one or more values due to out-of-bounds coordinate in a dimension without under/overflow tracking
fill() discarded one or more values due to out-of-bounds coordinate in a dimension without under/overflow tracking
fill() discarded one or more values due to out-of-bounds coordinate in a dimension without under/overflow tracking


Read in the detector response and orientation file. The orientation is cut down to the time interval of the source.

In [ ]:
response_file = data_path / 'ResponseContinuum.o3.pol.e200_10000.b4.p12.relx.s10396905069491.m420.filtered.binnedpolarization.11D.h5'
dr = FullDetectorResponse.open(response_file)

sc_orientation = SpacecraftHistory.open(data_path/'DC4_final_530km_3_month_with_slew_1sbins_GalacticEarth_SAA.fits', tstart=Time(1835493492.2, format = 'unix'), tstop=Time(1835493492.8, format = 'unix'))

Define the GRB position and spectrum.

In [7]:
source_direction = SkyCoord(l=23.53, b=-53.44, frame='galactic', unit=u.deg)

a = 100. * u.keV
b = 10000. * u.keV
alpha = -0.7368949
beta = -2.095031
ebreak = 622.389 * u.keV
K = 300. / u.cm / u.cm / u.s

spectrum = Band_Eflux(a = a.value,
                      b = b.value,
                      alpha = alpha,
                      beta = beta,
                      E0 = ebreak.value,
                      K = K.value)

spectrum.a.unit = a.unit
spectrum.b.unit = b.unit
spectrum.E0.unit = ebreak.unit
spectrum.K.unit = K.unit

Define initial values of polarization level and angle and convert to Stokes parameters, fix the spectral parameters to their true values, and create the source model.

In [ ]:
polarization = LinearPolarization(80, 150) # polarization level (percentage out of 100), polarization angle (degrees)

Q = polarization.degree.value / 100. * np.cos(2. * polarization.angle.value * np.pi / 180.)
U = polarization.degree.value / 100. * np.sin(2. * polarization.angle.value * np.pi / 180.)
polarization = StokesPolarization(Q=Constant(k=Q), U=Constant(k=U))
spectral_component = SpectralComponent('grb', spectrum, polarization)

source = PointSource('source',                                 # Name of source (arbitrary, but needs to be unique)
                     l = source_direction.l.deg,               # Longitude (deg)
                     b = source_direction.b.deg,               # Latitude (deg)
                     components = [spectral_component])        # Spectral model

source.components['grb'].shape.K.fix = True
source.components['grb'].shape.E0.fix = True
source.components['grb'].shape.alpha.fix = True
source.components['grb'].shape.beta.fix = True

source.components['grb'].polarization.Q.value.k.min_value = -10.
source.components['grb'].polarization.Q.value.k.max_value = 10.
source.components['grb'].polarization.U.value.k.min_value = -10.
source.components['grb'].polarization.U.value.k.max_value = 10.

model = Model(source)

### Polarization fit in ICRS frame

Instantiate the COSI 3ML plugin, combine with the model in a JointLikelihood object, then perform maximum likelihood fit.

In [ ]:
data = EmCDSBinnedData(grb_background.binned_data.project('Em', 'Phi', 'PsiChi'))

total_bkg = background_before.binned_data.project('Em', 'Phi', 'PsiChi') + background_after.binned_data.project('Em', 'Phi', 'PsiChi')
bkg_dist = {'total_bkg':total_bkg+sys.float_info.min}
bkg = FreeNormBinnedBackground(bkg_dist, sc_history = sc_orientation, copy = False)

instrument_response = BinnedInstrumentResponse(dr, data)

psr = BinnedThreeMLPointSourceResponse(data = data,
                                       instrument_response = instrument_response,
                                       sc_history = sc_orientation,
                                       energy_axis = dr.axes['Ei'],
                                       polarization_axis = dr.axes['Pol'],
                                       nside = 2*data.axes['PsiChi'].nside)

response = BinnedThreeMLModelFolding(data = data, point_source_response = psr)

like_fun = PoissonLikelihood(data, response, bkg)

cosi = ThreeMLPluginInterface('cosi',
                              like_fun,
                              response,
                              bkg)

cosi.bkg_parameter['total_bkg'] = Parameter('total_bkg',  # background parameter
                                            0.0016,  # initial value of parameter
                                            min_value=0,  # minimum value of parameter
                                            max_value=100,  # maximum value of parameter
                                            delta=0.05,  # initial step used by fitting engine
                                            unit = u.Hz)
cosi.bkg_parameter['total_bkg'].fix = True

plugins = DataList(cosi)

like = JointLikelihood(copy.deepcopy(model), plugins, verbose=False)

_ = like.fit()

fitted_polarization = to_linear_polarization(like.results.optimized_model.source.spectrum.grb.polarization)

print(f'Polarization level: {fitted_polarization.degree.value}%, Polarization angle: {fitted_polarization.angle.value} deg')

18:35:05 INFO      set the minimizer to minuit                                             ]8;id=761983;file:///Users/eneights/software/miniforge3/envs/cosipy-develop/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=2766;file:///Users/eneights/software/miniforge3/envs/cosipy-develop/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

Best fit values:

,result,unit
parameter,,
source.spectrum.grb.polarization.Q.Constant.k,(-2 +/- 7) x 10^-1,
source.spectrum.grb.polarization.U.Constant.k,0.1 +/- 2.8,


Correlation matrix:

1.00,1.00
1.00,1.00


Values of -log(likelihood) at the minimum:

,-log(likelihood)
cosi,21753.18961182623
total,21753.18961182623


Values of statistical measures:

,statistical measures
AIC,43510.379353865035
BIC,43529.24178660432


Polarization level: 23.259633137560837%, Polarization angle: 83.27827204679078 deg


### Minimum detectable polarization

Calculate the minimum detectable polarization by simulating an unpolarized source otherwise equivalent to the source being analyzed a large number (~10,000) of times, and fitting the polarization each time. The minimum detectable polarization at the 99% confidence level is the 99th percentile of the distribution of fitted polarization fractions. This currently takes a long time to run, so this only runs 100 simulations which leads to a less accurate result.

In [ ]:
n = 100

spectral_component_mdp = SpectralComponent('grb_mdp', spectrum, polarization)

source_mdp = PointSource('source',                               
                         l = source_direction.l.deg,        
                         b = source_direction.b.deg,
                         components = [spectral_component_mdp])   

source_mdp.components['grb_mdp'].shape.K.fix = True
source_mdp.components['grb_mdp'].shape.E0.fix = True
source_mdp.components['grb_mdp'].shape.alpha.fix = True
source_mdp.components['grb_mdp'].shape.beta.fix = True

model_mdp = Model(source_mdp)

bkg_parameter = Parameter('total_bkg',
                          0.0016,
                          min_value=0,
                          max_value=100,
                          delta=0.05,
                          unit = u.Hz,
                          free = False)

mdp = compute_mdp(n, model_mdp, bkg, bkg_parameter, sc_orientation, response_file)

18:35:45 INFO      set the minimizer to minuit                                             ]8;id=507018;file:///Users/eneights/software/miniforge3/envs/cosipy-develop/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=654983;file:///Users/eneights/software/miniforge3/envs/cosipy-develop/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

18:35:46 INFO      set the minimizer to minuit                                             ]8;id=379317;file:///Users/eneights/software/miniforge3/envs/cosipy-develop/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=789131;file:///Users/eneights/software/miniforge3/envs/cosipy-develop/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

18:35:47 INFO      set the minimizer to minuit                                             ]8;id=921328;file:///Users/eneights/software/miniforge3/envs/cosipy-develop/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=611874;file:///Users/eneights/software/miniforge3/envs/cosipy-develop/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

18:35:48 INFO      set the minimizer to minuit                                             ]8;id=559452;file:///Users/eneights/software/miniforge3/envs/cosipy-develop/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=892132;file:///Users/eneights/software/miniforge3/envs/cosipy-develop/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

18:35:49 INFO      set the minimizer to minuit                                             ]8;id=869898;file:///Users/eneights/software/miniforge3/envs/cosipy-develop/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=866623;file:///Users/eneights/software/miniforge3/envs/cosipy-develop/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

...

18:37:44 INFO      set the minimizer to minuit                                             ]8;id=876684;file:///Users/eneights/software/miniforge3/envs/cosipy-develop/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=290286;file:///Users/eneights/software/miniforge3/envs/cosipy-develop/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

18:37:45 WARNING   59.599999999999994 percent of samples have been thrown away because     ]8;id=924711;file:///Users/eneights/software/miniforge3/envs/cosipy-develop/lib/python3.12/site-packages/threeML/analysis_results.py\analysis_results.py]8;;\:]8;id=774642;file:///Users/eneights/software/miniforge3/envs/cosipy-develop/lib/python3.12/site-packages/threeML/analysis_results.py#1645\1645]8;;\
                  they failed the constraints on the parameters. This results might not be                         
                  suitable for error propagation. Enlarge the boundaries until you loose                           
                  less than 1 percent of the samples.                                                              

         INFO      set the minimizer to minuit                                             ]8;id=858812;file:///Users/eneights/software/miniforge3/envs/cosipy-develop/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=732113;file:///Users/eneights/software/miniforge3/envs/cosipy-develop/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

18:37:46 INFO      set the minimizer to minuit                                             ]8;id=964026;file:///Users/eneights/software/miniforge3/envs/cosipy-develop/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py\joint_likelihood.py]8;;\:]8;id=491472;file:///Users/eneights/software/miniforge3/envs/cosipy-develop/lib/python3.12/site-packages/threeML/classicMLE/joint_likelihood.py#1017\1017]8;;\

0/100 fits failed


In [ ]:
print(f'Minimum detectable polarization: {mdp:.2f}%')

Minimum detectable polarization: 8.11%
